# ทำความสะอาดข้อมูล ขั้นที่ 5/5: Data Dictionary + ไฟล์สะอาดสุดท้าย + สรุป Pipeline

| ขั้น | เรื่อง | สถานะ |
|---|---|---|
| 1 | โครงสร้าง คีย์หลัก ข้อมูลซ้ำ ขอบเขต | ✅ |
| 2 | ค่าว่าง | ✅ |
| 3 | ค่าผิดปกติ | ✅ |
| 4 | มาตรฐานรูปแบบ + ความสอดคล้อง | ✅ |
| **5 (ไฟล์นี้)** | **Data Dictionary + ส่งออก + ทำซ้ำได้** | เกณฑ์ 3.5.1.3 (5 คะแนน) |

เกณฑ์ 3.5.1.3 มี 3 ข้อ และขั้นนี้ทำครบทั้ง 3
1. **Data Dictionary ครบถ้วน**: ชื่อฟิลด์, คำอธิบาย, ชนิด, หน่วย/โดเมน, ตัวอย่างค่า
2. **Pipeline ทำซ้ำได้**: ลำดับขั้น, ชื่อโน้ตบุ๊ก, คำสั่งรัน, dependency/เวอร์ชัน
3. **แพ็กไฟล์มาตรฐาน**: รูปแบบเปิดได้ (CSV/Excel), โครงสร้างโฟลเดอร์ชัด, ไฟล์ตัวอย่างเล็กสำหรับทดสอบเร็ว

**อินพุต:** `clean_output/clean_step4.csv`

**เอาต์พุต (โฟลเดอร์ใหม่ `final/` = ของที่ใช้สร้างแดชบอร์ดและส่งกรรมการ):**
```
final/
├── energy_clean.csv          ← ข้อมูลสะอาดฉบับสุดท้าย (ใช้สร้างแดชบอร์ด)
├── energy_clean.xlsx         ← ฉบับ Excel: ข้อมูล + Data Dictionary + Log ในไฟล์เดียว
├── energy_clean_sample.csv   ← ไฟล์ตัวอย่างเล็ก (อาเซียน 2 ปี) ทดสอบเร็ว
├── data_dictionary.csv       ← พจนานุกรมข้อมูล
├── data_quality_summary.csv  ← สรุปคุณภาพข้อมูลก่อน–หลัง
├── requirements.txt          ← เวอร์ชันไลบรารี
└── README.md                 ← วิธีรัน pipeline ทั้งหมด
```

> รัน **Kernel → Restart Kernel and Run All Cells** (ต้องรันขั้นที่ 4 สำเร็จก่อน)

## 5.1 โหลดผลจากขั้นที่ 4

In [ ]:
import os, sys, datetime
import pandas as pd
import numpy as np
import matplotlib

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

OUT_DIR = "clean_output"
FINAL_DIR = "final"
IN_FILE = os.path.join(OUT_DIR, "clean_step4.csv")
LOG_FILE = os.path.join(OUT_DIR, "cleaning_log.csv")
if not os.path.exists(IN_FILE):
    raise FileNotFoundError("ไม่พบ clean_output/clean_step4.csv ให้รัน clean_step4_standardize.ipynb ให้สำเร็จก่อน")
os.makedirs(FINAL_DIR, exist_ok=True)

df = pd.read_csv(IN_FILE)
for c in ["country", "iso_code", "region"]:
    df[c] = df[c].astype("category")
df["population"] = df["population"].astype("Int64")
df["gdp"] = df["gdp"].astype("Int64")

log_old = pd.read_csv(LOG_FILE)
log_old = log_old[~log_old["step"].str.startswith("5-")]
print("ข้อมูลจากขั้นที่ 4:", df.shape, "| ประเทศ:", df["country"].nunique(),
      "| ปี:", df["year"].min(), "-", df["year"].max())

## 5.2 ตรวจขั้นสุดท้าย (Data Contract)
ก่อนส่งออก ตรวจ "สัญญา" ที่ข้อมูลสะอาดต้องเป็นจริงทุกข้อ ถ้าข้อไหนไม่ผ่าน โค้ดจะหยุดทันที (ใช้ `assert`) ไม่ปล่อยให้ข้อมูลเสียไปถึงแดชบอร์ด

In [ ]:
SHARE = [c for c in df.columns if "share_elec" in c]
TWH = [c for c in df.columns if c.endswith("_electricity") and c != "per_capita_electricity"] + \
      ["electricity_generation", "electricity_demand", "primary_energy_consumption"]

contract = {
    "คีย์ country+year ไม่ซ้ำ":              not df.duplicated(["country", "year"]).any(),
    "คีย์ไม่มีค่าว่าง":                         df[["country", "iso_code", "year"]].notna().all().all(),
    "iso_code เป็นตัวอักษร 3 ตัวทุกแถว":       df["iso_code"].astype(str).str.fullmatch(r"[A-Z]{3}").all(),
    "ทุกประเทศมีทวีป":                         df["region"].notna().all(),
    "ปีอยู่ในช่วง 2000-2022":                   df["year"].between(2000, 2022).all(),
    "สัดส่วนอยู่ใน 0-100":                      (df[SHARE].isna() | ((df[SHARE] >= 0) & (df[SHARE] <= 100))).all().all(),
    "ปริมาณไฟฟ้า/พลังงานไม่ติดลบ":             (df[TWH].isna() | (df[TWH] >= 0)).all().all(),
    "ไฟฟ้าทั้งหมด = ฟอสซิล + คาร์บอนต่ำ":       ((df["fossil_electricity"] + df["low_carbon_electricity"]
                                               - df["electricity_generation"]).abs().dropna() < 0.01).all(),
    "ฟีเจอร์สำคัญว่าง <= 5%":                   (df[["electricity_generation", "per_capita_electricity",
                                                  "renewables_share_elec", "fossil_share_elec"]].isna().mean() <= 0.05).all(),
}
for rule, ok in contract.items():
    print("✅" if ok else "❌", rule)
assert all(contract.values()), "มีข้อที่ไม่ผ่าน ห้ามส่งออก ให้กลับไปตรวจขั้นก่อนหน้า"
print("\nผ่านทุกข้อ พร้อมส่งออก")

## 5.3 สร้าง Data Dictionary
เขียนคำอธิบายภาษาไทยให้ทุกคอลัมน์ แล้วให้โค้ด **ดึงชนิดข้อมูล หน่วย ช่วงค่า ตัวอย่างค่า และ % ว่าง จากข้อมูลจริงอัตโนมัติ** (พิมพ์เองจะผิดง่ายและไม่ตรงกับข้อมูล)
คอลัมน์ "ที่มา" บอกว่าค่ามาจากแหล่งข้อมูลโดยตรง หรือเราสร้าง/คำนวณขึ้นในขั้นไหน ทำให้ตรวจสอบย้อนกลับได้

In [ ]:
# (คำอธิบาย, หน่วย, ที่มา)
SRC = "Our World in Data"
DESC = {
    "country": ("ชื่อประเทศ/เขตปกครอง (ภาษาอังกฤษ)", "text", SRC),
    "iso_code": ("รหัสประเทศ ISO 3166-1 alpha-3", "text", SRC + " (Kosovo เติม XKX ในขั้น 1)"),
    "region": ("ทวีป", "Africa/Asia/Europe/North America/Oceania/South America", "สร้างในขั้น 4 จาก iso_code"),
    "is_asean": ("เป็นสมาชิกอาเซียน", "True/False", "สร้างในขั้น 4"),
    "year": ("ปี ค.ศ.", "year", SRC),
    "year_complete": ("ปีนี้มีข้อมูลไฟฟ้า >= 90% ของประเทศ (False = ข้อมูลเบื้องต้น ใช้รวมยอดโลกไม่ได้)", "True/False", "สร้างในขั้น 4"),
    "gdp_available": ("มีค่า GDP ในแถวนี้ (แหล่งข้อมูล GDP สิ้นสุดปี 2018)", "True/False", "สร้างในขั้น 2"),
    "is_outlier_pc_elec": ("ไฟฟ้าต่อคนสูง/ต่ำผิดปกติเมื่อเทียบประเทศอื่นในปีเดียวกัน (IQR) แต่เป็นค่าจริง", "True/False", "สร้างในขั้น 3"),
    "low_precision": ("ผลิตไฟฟ้า < 0.1 TWh ตัวเลขถูกปัด สัดส่วน/ความเข้มคาร์บอนอาจเพี้ยน", "True/False", "สร้างในขั้น 3"),
    "population": ("จำนวนประชากร", "persons", SRC),
    "gdp": ("ผลิตภัณฑ์มวลรวมในประเทศ ปรับค่าครองชีพ", "international-$ (2011 prices)", SRC),
    "electricity_generation": ("ไฟฟ้าที่ผลิตได้ทั้งหมดในประเทศ", "TWh", SRC),
    "electricity_demand": ("ความต้องการไฟฟ้า = ผลิตเอง + นำเข้าสุทธิ", "TWh", SRC + " + คำนวณเติมในขั้น 2"),
    "net_elec_imports": ("ไฟฟ้านำเข้าสุทธิ (ติดลบ = ส่งออกมากกว่านำเข้า)", "TWh", SRC + " + คำนวณเติมในขั้น 2"),
    "per_capita_electricity": ("ไฟฟ้าที่ผลิตต่อประชากร 1 คน ต่อปี", "kWh/person", SRC + " + คำนวณเติมในขั้น 2"),
    "carbon_intensity_elec": ("ความเข้มคาร์บอน: CO2 ที่ปล่อยต่อการผลิตไฟฟ้า 1 kWh (ยิ่งต่ำยิ่งสะอาด)", "gCO2/kWh", SRC),
    "greenhouse_gas_emissions": ("ก๊าซเรือนกระจกจากการผลิตไฟฟ้า", "MtCO2e", SRC),
    "primary_energy_consumption": ("การใช้พลังงานปฐมภูมิทั้งหมด (ไฟฟ้า + ขนส่ง + ความร้อน ฯลฯ)", "TWh", SRC),
    "energy_per_capita": ("การใช้พลังงานปฐมภูมิต่อคน", "kWh/person", SRC + " + คำนวณเติมในขั้น 2"),
}
FUEL_TH = {"coal": "ถ่านหิน", "gas": "ก๊าซธรรมชาติ", "oil": "น้ำมัน", "nuclear": "นิวเคลียร์", "hydro": "พลังน้ำ",
           "solar": "แสงอาทิตย์", "wind": "ลม", "biofuel": "ชีวมวล/เชื้อเพลิงชีวภาพ",
           "other_renewable_exc_biofuel": "หมุนเวียนอื่น (เช่น ความร้อนใต้พิภพ, คลื่น) ไม่รวมชีวมวล",
           "fossil": "ฟอสซิลรวม (ถ่านหิน+ก๊าซ+น้ำมัน)", "renewables": "หมุนเวียนรวม (น้ำ+แสงอาทิตย์+ลม+ชีวมวล+อื่น)",
           "low_carbon": "คาร์บอนต่ำรวม (หมุนเวียน+นิวเคลียร์)"}
for f, th in FUEL_TH.items():
    DESC[f"{f}_electricity"] = (f"ไฟฟ้าที่ผลิตจาก{th}", "TWh", SRC + " + คำนวณเติมในขั้น 2")
    share = "other_renewables_share_elec_exc_biofuel" if f == "other_renewable_exc_biofuel" else f"{f}_share_elec"
    DESC[share] = (f"สัดส่วนไฟฟ้าจาก{th} ต่อไฟฟ้าทั้งหมด", "% (0-100)", SRC + " + คำนวณเติมในขั้น 2")

no_desc = [c for c in df.columns if c not in DESC]
assert not no_desc, f"คอลัมน์ที่ยังไม่มีคำอธิบาย: {no_desc}"
print("มีคำอธิบายครบทุกคอลัมน์ ✅", len(DESC), "คอลัมน์")

In [ ]:
def type_label(s):
    if isinstance(s.dtype, pd.CategoricalDtype): return "category (text)"
    if pd.api.types.is_bool_dtype(s): return "boolean"
    if pd.api.types.is_integer_dtype(s): return "integer"
    if pd.api.types.is_float_dtype(s): return "float"
    return "text"

def domain_of(s):
    if isinstance(s.dtype, pd.CategoricalDtype) or s.dtype == object:
        return f"{s.nunique()} ค่าไม่ซ้ำ"
    if pd.api.types.is_bool_dtype(s):
        return f"True {int(s.sum()):,} แถว / False {int((~s).sum()):,} แถว"
    return f"{s.min():,} ถึง {s.max():,}"

def example_of(col):
    th = df.loc[(df["country"] == "Thailand") & (df["year"] == 2021), col]   # ใช้ไทยปี 2021 เป็นตัวอย่าง
    v = th.iloc[0] if len(th) and pd.notna(th.iloc[0]) else df[col].dropna().iloc[0]
    return f"{v:,}" if isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, (bool, np.bool_)) else str(v)

dictionary = pd.DataFrame([{
    "no": i + 1,
    "column": c,
    "description_th": DESC[c][0],
    "type": type_label(df[c]),
    "unit_or_domain": DESC[c][1],
    "range_in_data": domain_of(df[c]),
    "example (Thailand 2021)": example_of(c),
    "missing_pct": round(df[c].isna().mean() * 100, 2),
    "source": DESC[c][2],
} for i, c in enumerate(df.columns)])

dictionary.to_csv(os.path.join(FINAL_DIR, "data_dictionary.csv"), index=False, encoding="utf-8-sig")
dictionary

## 5.4 สรุปคุณภาพข้อมูลก่อน–หลัง (ทั้ง pipeline)
ตารางนี้คือ **หลักฐานชิ้นเดียวที่สรุปงานทั้ง 5 ขั้น** เหมาะใส่สไลด์นำเสนอ

In [ ]:
raw = pd.read_csv([f for f in sorted(os.listdir(".")) if f.startswith("World_Energy_Consumption") and f.endswith(".csv")][0])
KEYF = ["electricity_generation", "per_capita_electricity", "renewables_share_elec", "fossil_share_elec"]

def quality(d, label):
    return pd.Series({
        "แถว": f"{len(d):,}",
        "คอลัมน์": d.shape[1],
        "ประเทศ/ชื่อ": d["country"].nunique(),
        "ช่วงปี": f"{d['year'].min()}-{d['year'].max()}",
        "กลุ่มภูมิภาคปน (นับซ้ำได้)": int(d["iso_code"].isna().sum() > 0) and d.loc[d["iso_code"].isna(), "country"].nunique(),
        "คีย์ซ้ำ (%)": round(d.duplicated(["country", "year"]).mean() * 100, 2),
        "ค่าว่างเฉลี่ยทั้งตาราง (%)": round(d.isna().mean().mean() * 100, 1),
        "ค่าว่างฟีเจอร์สำคัญ (%)": round(d[KEYF].isna().mean().mean() * 100, 2),
        "0 ปลอม (ผลิตไฟฟ้า=0 แต่มีประชากร)": int(((d["electricity_generation"] == 0) & (d["population"] > 0)).sum()),
    }, name=label)

summary = pd.concat([quality(raw, "ข้อมูลดิบ"), quality(df, "หลังทำความสะอาด")], axis=1)
summary.to_csv(os.path.join(FINAL_DIR, "data_quality_summary.csv"), encoding="utf-8-sig")
summary

In [ ]:
# log ทุกขั้นที่ทำมา
log = pd.read_csv(LOG_FILE)
log[["step", "action", "rows_before", "rows_after", "cols_before", "cols_after"]]

## 5.5 ส่งออกไฟล์สุดท้าย
- **CSV** (`utf-8-sig` เปิดใน Excel แล้วภาษาไทยไม่เพี้ยน) ใช้สร้างแดชบอร์ด
- **Excel** รวม 4 ชีต: ข้อมูล, Data Dictionary, สรุปคุณภาพ, Log ให้กรรมการเปิดดูไฟล์เดียวจบ
- **ไฟล์ตัวอย่างเล็ก** อาเซียน 2 ปี ใช้ทดสอบแดชบอร์ดเร็ว ๆ โดยไม่ต้องโหลดข้อมูลทั้งหมด

In [ ]:
FINAL_CSV = os.path.join(FINAL_DIR, "energy_clean.csv")
df.to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")

sample = df[df["is_asean"] & df["year"].isin([2000, 2021])]
sample.to_csv(os.path.join(FINAL_DIR, "energy_clean_sample.csv"), index=False, encoding="utf-8-sig")

try:
    with pd.ExcelWriter(os.path.join(FINAL_DIR, "energy_clean.xlsx")) as xw:
        df.to_excel(xw, sheet_name="data", index=False)
        dictionary.to_excel(xw, sheet_name="data_dictionary", index=False)
        summary.to_excel(xw, sheet_name="quality_summary")
        log.to_excel(xw, sheet_name="cleaning_log", index=False)
    print("สร้าง energy_clean.xlsx แล้ว")
except ModuleNotFoundError:
    print("⚠️ ไม่มีไลบรารี openpyxl จึงข้ามไฟล์ Excel (ติดตั้งด้วย: pip install openpyxl แล้วรันเซลล์นี้ใหม่)")

print(f"energy_clean.csv: {len(df):,} แถว | ตัวอย่าง: {len(sample)} แถว")

In [ ]:
# ทดสอบว่าไฟล์ที่ส่งออก อ่านกลับมาแล้วได้ข้อมูลเหมือนเดิม (round-trip test)
back = pd.read_csv(FINAL_CSV)
same_shape = back.shape == df.shape
same_total = abs(back["electricity_generation"].sum() - df["electricity_generation"].sum()) < 1e-6
same_th = back.loc[(back["country"] == "Thailand") & (back["year"] == 2021), "renewables_share_elec"].iloc[0] == \
          df.loc[(df["country"] == "Thailand") & (df["year"] == 2021), "renewables_share_elec"].iloc[0]
print("ขนาดตรงกัน:", "✅" if same_shape else "❌", back.shape)
print("ยอดรวมไฟฟ้าตรงกัน:", "✅" if same_total else "❌")
print("ค่าไทย 2021 ตรงกัน:", "✅" if same_th else "❌")

## 5.6 Pipeline ทำซ้ำได้: requirements.txt + README.md
กรรมการต้องรันซ้ำได้บนเครื่องอื่น จึงต้องบอก **เวอร์ชันไลบรารี** และ **ลำดับ/คำสั่งรัน**

In [ ]:
import importlib.metadata as im
pkgs = ["pandas", "numpy", "matplotlib", "openpyxl", "jupyterlab", "notebook", "nbconvert"]
req = []
for p in pkgs:
    try: req.append(f"{p}=={im.version(p)}")
    except im.PackageNotFoundError: pass
with open(os.path.join(FINAL_DIR, "requirements.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(req) + "\n")
print(f"Python {sys.version.split()[0]}")
print("\n".join(req))

In [ ]:
TODAY = datetime.date.today().isoformat()
n_countries = df["country"].nunique()
latest_complete = int(df.loc[df["year_complete"], "year"].max())

readme = f"""# World Energy Consumption: Data Cleaning Pipeline

สร้างเมื่อ: {TODAY} | Python {sys.version.split()[0]} | pandas {pd.__version__}

## ข้อมูล
- แหล่ง: Our World in Data, Energy dataset (`World_Energy_Consumption.csv`)
- หลังทำความสะอาด: {len(df):,} แถว, {df.shape[1]} คอลัมน์, {n_countries} ประเทศ, ปี {df["year"].min()}-{df["year"].max()}
- ปีล่าสุดที่ข้อมูลครบ: {latest_complete} (ปี 2022 = ข้อมูลเบื้องต้น ดูคอลัมน์ `year_complete`)
- 1 แถว = 1 ประเทศ ใน 1 ปี (คีย์หลัก: country + year)

## โครงสร้างโฟลเดอร์
```
energy_project/
├── World_Energy_Consumption.csv     ข้อมูลดิบ (ห้ามแก้)
├── step1_questions.ipynb            ตั้งคำถาม
├── clean_step1_structure.ipynb      ขั้น 1 โครงสร้าง คีย์ ข้อมูลซ้ำ ขอบเขต
├── clean_step2_missing.ipynb        ขั้น 2 ค่าว่าง
├── clean_step3_outliers.ipynb       ขั้น 3 ค่าผิดปกติ
├── clean_step4_standardize.ipynb    ขั้น 4 มาตรฐาน + ความสอดคล้อง
├── clean_step5_dictionary.ipynb     ขั้น 5 Data Dictionary + ส่งออก
├── clean_output/                    ผลระหว่างทาง + รายงานตรวจสอบทุกขั้น
└── final/                           ไฟล์สุดท้ายสำหรับแดชบอร์ด
```

## วิธีรัน (ต้องรันเรียงลำดับ 1 → 5)
ติดตั้งไลบรารี:
```
pip install -r final/requirements.txt
```
รันทั้งหมดด้วยคำสั่งเดียว (ในโฟลเดอร์ energy_project):
```
jupyter nbconvert --to notebook --execute --inplace clean_step1_structure.ipynb clean_step2_missing.ipynb clean_step3_outliers.ipynb clean_step4_standardize.ipynb clean_step5_dictionary.ipynb
```
หรือเปิดใน JupyterLab ทีละไฟล์ แล้วใช้ Kernel → Restart Kernel and Run All Cells

## สรุปแต่ละขั้น
| ขั้น | อินพุต | เอาต์พุต | สิ่งที่ทำ |
|---|---|---|---|
| 1 | ข้อมูลดิบ | clean_step1.csv, profile_raw.csv | ตรวจคีย์/ซ้ำ/ชนิด, ตัดกลุ่มภูมิภาค, เลือกปี 2000-2022 และ 37 คอลัมน์ |
| 2 | clean_step1.csv | clean_step2.csv, filled_cells.csv | 0 ปลอม → ค่าว่าง, ลบประเทศไม่มีข้อมูล, คำนวณเติมจากสูตร, ไม่เติม GDP หลัง 2018 |
| 3 | clean_step2.csv | clean_step3.csv, outlier_report.csv | กฎโดเมน, IQR รายปี, การกระโดดตามเวลา, ความแม่นยำประเทศเล็ก |
| 4 | clean_step3.csv | clean_step4.csv, consistency_report.csv, reconciliation.csv | หน่วย, ทศนิยม, ทวีป/อาเซียน, ความสอดคล้อง 6 ข้อ, กระทบยอด, ป้ายปีไม่ครบ |
| 5 | clean_step4.csv | final/* | Data contract, Data Dictionary, ส่งออก CSV/Excel/ตัวอย่าง |

## ข้อจำกัดที่ต้องรู้ก่อนใช้
- GDP มีถึงปี 2018 เท่านั้น
- ปี 2022 มีข้อมูลไม่ถึงครึ่งของประเทศ ห้ามใช้รวมยอดโลก
- ไฟฟ้าต่อคนเบ้ขวามาก ใช้มัธยฐานเป็นค่ากลางหลัก
- ประเทศที่ low_precision = True ไม่ควรใช้จัดอันดับสัดส่วน/ความเข้มคาร์บอน
"""
with open(os.path.join(FINAL_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme)
print(readme[:1200], "...")

## 5.7 บันทึก log ขั้นสุดท้าย และตรวจไฟล์ที่ส่งมอบ

In [ ]:
log_rows = [{"step": "5-export", "action": "ตรวจ data contract + ส่งออก final/ (CSV, Excel, sample, dictionary)",
             "rows_before": len(df), "rows_after": len(df), "cols_before": df.shape[1], "cols_after": df.shape[1],
             "reason": "ผ่าน data contract ทุกข้อ"}]
pd.concat([log_old, pd.DataFrame(log_rows)]).to_csv(LOG_FILE, index=False, encoding="utf-8-sig")

print("📁 final/")
for f in sorted(os.listdir(FINAL_DIR)):
    print(f"   ├── {f:<28} {os.path.getsize(os.path.join(FINAL_DIR, f))/1024:>8,.0f} KB")
print("\n📁 clean_output/ (หลักฐานระหว่างทาง)")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"   ├── {f}")

## 🎉 ทำความสะอาดข้อมูลครบ 5 ขั้นแล้ว

### เช็กลิสต์
- [ ] Data contract ข้อ 5.2 ขึ้น ✅ ทุกข้อ
- [ ] round-trip test ข้อ 5.5 ขึ้น ✅ ทั้ง 3 ข้อ
- [ ] เปิด `final/energy_clean.xlsx` ใน Excel แล้วเห็น 4 ชีต
- [ ] อ่าน `final/README.md` แล้วเข้าใจลำดับการรัน

### เกณฑ์ 3.5.1 (18 คะแนน) ทำครบแล้ว
| เกณฑ์ | หลักฐานอยู่ที่ |
|---|---|
| 3.5.1.1 ความครบถ้วน (7) | ขั้น 1: profile_raw.csv, คีย์, FK match rate · ขั้น 2: missing_before_after.csv · ขั้น 4: หน่วย |
| 3.5.1.2 ทำความสะอาด (6) | ขั้น 2: filled_cells.csv · ขั้น 3: outlier_report.csv · ขั้น 4: consistency_report.csv, reconciliation.csv |
| 3.5.1.3 Dictionary/Steps (5) | ขั้น 5: data_dictionary.csv, README.md, requirements.txt, ไฟล์ตัวอย่าง |

**ประโยคสรุปไว้ตอบกรรมการ:** "ข้อมูลทุกตัวบนแดชบอร์ดมาจาก energy_clean.csv ไฟล์เดียว ซึ่งผ่าน pipeline 5 โน้ตบุ๊กที่รันซ้ำได้ด้วยคำสั่งเดียว ทุกการเปลี่ยนแปลงอยู่ใน cleaning_log.csv และทุกคอลัมน์มีคำอธิบายใน Data Dictionary"

**ขั้นถัดไป: สร้างแดชบอร์ด** จาก `final/energy_clean.csv` (หน้า Overview → Detail, ตัวกรองประเทศ/ปี/ทวีป, ปุ่มส่งออก ตามเกณฑ์ 3.5.3–3.5.6)